# 09 · Contrato de aprendizaje y cierre técnico

Paso 30. La referencia utiliza una realización U; la alternativa PU agregará scores de varias realizaciones manteniendo P y evaluación. Los modelos se ajustarán en fase F. Un score P/U no es una probabilidad absoluta de encontrar oro.

In [1]:
from pathlib import Path
import sys
import importlib
import pandas as pd
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'src/geoau/evaluation.py').is_file())
if str(ROOT / 'src') not in sys.path: sys.path.insert(0, str(ROOT / 'src'))
from geoau import evaluation as ev
importlib.reload(ev)

RUN = ev.current_run(ROOT)
control = ev.finish_run(ROOT, RUN)
display(control)
display(ev.read_json(RUN / "learning_contract.json"))

{'estado_ejecucion': 'completada',
 'mode': 'diagnostic',
 'fase_e_cientifica_cerrada': False,
 'training_allowed': False,
 'prediction_allowed': False,
 'eligible_cells': 472548,
 'reviewed_positive_cells': 0,
 'selected_protocol_positive_cells': 666,
 'outer_folds': 5,
 'split_designs': 20,
 'sample_designs': 180,
 'minimum_gap_lower_bound_m': 5293.990370126274,
 'reasons_not_ready': ['No hay positivos revisados con soporte para este objetivo.',
  'Faltan deposit_id validados en los positivos.',
  'Faltan district_id validados en los positivos.',
  'D no contiene predictores aprobados para entrenamiento.',
  'Falta cartografía territorial de distritos: no basta con IDs en los indicios.',
  'Falta selección revisada de distritos para reserva final.',
  'Protocolo espacial pendiente de revisión.']}

{'mode': 'diagnostic',
 'reference': 'P frente a U; U no significa ausencia de oro',
 'baseline': 'realización 0 de cada ratio; elegir ratio solo dentro del bucle interno',
 'pu_alternative': 'bagging de realizaciones U; mismos P y marco de evaluación por split',
 'aggregation': 'media de scores de las realizaciones; no probabilidad absoluta de oro',
 'evaluation': 'todas las celdas elegibles con role=test en membresía; no muestrear el test',
 'holdout': 'no se generan muestras para entrenar/ajustar sobre la reserva',
 'weights': 'inversos de inclusión de U, no class_weight ni probabilidad de observación de P',
 'preprocessing': 'imputación, codificación, selección y escalado se ajustan en cada train interno',
 'model_fitting': 'corresponde a fase F; aquí solo contratos y diseños',
 'automatic_scar_or_prevalence_assumption': False}

## Condiciones para entrenar
La selección de variables, algoritmo, ratio e hiperparámetros pertenece al bucle interno. El externo evalúa ese procedimiento completo. La reserva solo se abre una vez fijadas las decisiones. Seleccionar modelos repetidamente por la reserva invalida su papel de prueba final.

In [2]:
ev.verify(RUN, ev.read_json(RUN / 'outputs_manifest.json'))
try:
    ev.assert_ready_for_training(ROOT, RUN)
    print('Protocolo aprobado para fase F.')
except ValueError as error:
    if control.get('training_allowed'): raise
    print(str(error))
    display(control['reasons_not_ready'])
print('Ejecución:', RUN)

Diseño diagnóstico/no aprobado: no usar como conjunto validado para entrenamiento.


['No hay positivos revisados con soporte para este objetivo.',
 'Faltan deposit_id validados en los positivos.',
 'Faltan district_id validados en los positivos.',
 'D no contiene predictores aprobados para entrenamiento.',
 'Falta cartografía territorial de distritos: no basta con IDs en los indicios.',
 'Falta selección revisada de distritos para reserva final.',
 'Protocolo espacial pendiente de revisión.']

Ejecución: C:\Users\Lenovo\Desktop\PROYECTO IA\Proyecto Con Luis\reports\fase_e\20260909T124128_606717Z


Para abandonar el modo diagnóstico hacen falta etiquetas revisadas, depósitos y distritos identificados, cartografía territorial de distritos, predictores aprobados en D y revisión del protocolo. Para el objetivo aluvial se requieren además cuencas. Cambiar únicamente `mode` no supera estos controles.